In [1]:
!git clone https://github.com/leefige/radik.git
%cd radik

Cloning into 'radik'...
remote: Enumerating objects: 340, done.
remote: Counting objects: 100% (340/340), done.
remote: Compressing objects: 100% (232/232), done.
remote: Total 340 (delta 110), reused 320 (delta 94), pack-reused 0 (from 0)
Receiving objects: 100% (340/340), 590.57 KiB | 15.14 MiB/s, done.
Resolving deltas: 100% (110/110), done.
/content/radik


In [2]:
!nvidia-smi
# For T4: SM_VERSION=75, A100: 80, L4: 89, V100: 70
SM_VERSION = "75"  # Change if you have different GPU

Wed Mar 18 14:13:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
print("Building RadiK...")
!cd radik && make clean && make SM_VERSION=75
print("✅ Build complete!")

Building RadiK...
rm -rf test_radik.out test_PQ.out test_ablation.out
/usr/local/cuda/bin/nvcc -std=c++14 -O3 -arch=sm_75 -lcurand test/test_radik.cu -o test_radik.out -Xptxas=-v
ptxas info    : 29 bytes gmem, 232 bytes cmem[4]
ptxas info    : Compiling entry function '_ZN10radix_topk7bitonic16bitonicSort_4096ILb1EfiEEvPT0_PT1_iPKi' for 'sm_75'
ptxas info    : Function properties for _ZN10radix_topk7bitonic16bitonicSort_4096ILb1EfiEEvPT0_PT1_iPKi
    0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
ptxas info    : Used 64 registers, used 1 barriers, 32768 bytes smem, 384 bytes cmem[0]
ptxas info    : Compile time = 696.253 ms
ptxas info    : Compiling entry function '_ZN10radix_topk7bitonic16bitonicSort_2048ILb1EfiEEvPT0_PT1_iPKi' for 'sm_75'
ptxas info    : Function properties for _ZN10radix_topk7bitonic16bitonicSort_2048ILb1EfiEEvPT0_PT1_iPKi
    0 bytes stack frame, 0 bytes spill stores, 0 bytes spill loads
ptxas info    : Used 59 registers, used 1 barriers, 16384 byte

In [4]:
# =============================================================================
# RadiK & PyTorch topk Benchmark
# =============================================================================

import os
import numpy as np
import torch
import torch.nn.functional as F

RADIK_BIN = "./radik/test_radik.out"

def run_radik(N_pow2, K, warmup=True):
    """Run with warmup to avoid JIT"""
    cmd = f"{RADIK_BIN} 1 {N_pow2} {K} 0 0 0 1 1 1"

    # Warmup run
    if warmup:
        with os.popen(cmd, 'r') as fout:
            for line in fout:
                if "elapsed:" in line:
                    break

    # Real run
    with os.popen(cmd, 'r') as fout:
        for line in fout:
            if "elapsed:" in line:
                return float(line.split()[1])
    return None

def run_pytorch_topk(N, K, device='cuda', dtype=torch.float32, warmup=True):
    """
    Benchmark PyTorch torch.topk on GPU
    Uses CUDA events for accurate GPU timing (includes kernel launch + execution)
    """
    # Create random data on GPU
    x = torch.randn(N, device=device, dtype=dtype)

    # Warmup runs (important for CUDA context initialization and caching)
    if warmup:
        for _ in range(10):
            _ = torch.topk(x, K, largest=True, sorted=True)
        torch.cuda.synchronize(device)

    # Ensure GPU is ready
    torch.cuda.synchronize(device)

    # Create CUDA events for timing
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)
    start_event.record()
    values, indices = torch.topk(x, K, largest=True, sorted=True)
    end_event.record()

    torch.cuda.synchronize(device)  # Wait for completion
    elapsed_ms = start_event.elapsed_time(end_event)

    return elapsed_ms

# Configuration
N_powers = [21, 23, 25, 27, 29]  # Only odd powers
K_values = [16, 32, 64, 128, 256, 512, 1024, 2048, 4096]

print("RadiK vs PyTorch Top-K Benchmark")
print("=" * 80)

# Check PyTorch CUDA availability
if not torch.cuda.is_available():
    raise RuntimeError("CUDA not available for PyTorch benchmarking")

device = torch.device('cuda')
print(f"PyTorch device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print("=" * 80)

# Storage for results
radik_results = []
pytorch_results = []

for N_pow2 in N_powers:
    N = 1 << N_pow2
    print(f"\nN = 2^{N_pow2} ({N:,})")

    radik_row = []
    pytorch_row = []

    for K in K_values:
        # Skip if K > N (though shouldn't happen with these configs)
        if K > N:
            radik_row.append(np.nan)
            pytorch_row.append(np.nan)
            print(f"  K={K:4d}: SKIPPED (K > N)")
            continue

        # Run RadiK
        t_radik = run_radik(N_pow2, K, warmup=True)
        radik_row.append(t_radik if t_radik else np.nan)

        # Run PyTorch
        t_pytorch = run_pytorch_topk(N, K, device=device, warmup=True)
        pytorch_row.append(t_pytorch)

        # Print comparison
        if t_radik and t_pytorch:
            speedup = t_pytorch / t_radik
            print(f"  K={K:4d}: RadiK={t_radik:>7.2f} ms | PyTorch={t_pytorch:>7.2f} ms | Speedup={speedup:>5.2f}x")
        else:
            print(f"  K={K:4d}: RadiK={'FAILED' if not t_radik else f'{t_radik:.2f} ms'} | PyTorch={t_pytorch:.2f} ms")

    radik_results.append(radik_row)
    pytorch_results.append(pytorch_row)

# Convert to numpy arrays
radik_results = np.array(radik_results)
pytorch_results = np.array(pytorch_results)
np.save('radik_timings.npy', radik_results)
np.save('pytorch_timings.npy', pytorch_results)

# Print comparison table
print("\n" + "=" * 100)
print("SUMMARY TABLE (milliseconds)")
print("=" * 100)

print(f"\n{'N':>15} | {'K':>6} | {'RadiK':>10} | {'PyTorch':>10} | {'Speedup':>8}")
print("-" * 100)

for i, N_pow2 in enumerate(N_powers):
    N = 1 << N_pow2
    for j, K in enumerate(K_values):
        r_time = radik_results[i, j]
        p_time = pytorch_results[i, j]

        if not np.isnan(r_time) and not np.isnan(p_time) and r_time > 0:
            speedup = p_time / r_time
            print(f"2^{N_pow2:>2} ({N:>10,}) | {K:>6} | {r_time:>10.2f} | {p_time:>10.2f} | {speedup:>8.2f}x")
        else:
            print(f"2^{N_pow2:>2} ({N:>10,}) | {K:>6} | {'N/A':>10} | {'N/A':>10} | {'N/A':>8}")

# Print RadiK-only table (original format)
print("\n" + "=" * 90)
print("RadiK Results Only")
print(f"{'N\\K':>12}", ''.join(f"{K:>9}" for K in K_values))
print("-" * 90)
for i, N_pow2 in enumerate(N_powers):
    N = 1 << N_pow2
    print(f"2^{N_pow2} ({N:>10,}): ", end='')
    for val in radik_results[i]:
        print(f"{val:>9.2f}" if not np.isnan(val) else f"{'N/A':>9}", end='')
    print()

# Print PyTorch-only table
print("\n" + "=" * 90)
print("PyTorch Results Only")
print(f"{'N\\K':>12}", ''.join(f"{K:>9}" for K in K_values))
print("-" * 90)
for i, N_pow2 in enumerate(N_powers):
    N = 1 << N_pow2
    print(f"2^{N_pow2} ({N:>10,}): ", end='')
    for val in pytorch_results[i]:
        print(f"{val:>9.2f}" if not np.isnan(val) else f"{'N/A':>9}", end='')
    print()

RadiK vs PyTorch Top-K Benchmark
PyTorch device: cuda
GPU: Tesla T4

N = 2^21 (2,097,152)
  K=  16: RadiK=   4.18 ms | PyTorch=   0.55 ms | Speedup= 0.13x
  K=  32: RadiK=   3.19 ms | PyTorch=   0.44 ms | Speedup= 0.14x
  K=  64: RadiK=   3.08 ms | PyTorch=   0.37 ms | Speedup= 0.12x
  K= 128: RadiK=   3.10 ms | PyTorch=   0.38 ms | Speedup= 0.12x
  K= 256: RadiK=   3.16 ms | PyTorch=   0.41 ms | Speedup= 0.13x
  K= 512: RadiK=   3.09 ms | PyTorch=   0.38 ms | Speedup= 0.12x
  K=1024: RadiK=   3.10 ms | PyTorch=   0.39 ms | Speedup= 0.13x
  K=2048: RadiK=   3.19 ms | PyTorch=   0.41 ms | Speedup= 0.13x
  K=4096: RadiK=   3.43 ms | PyTorch=   0.49 ms | Speedup= 0.14x

N = 2^23 (8,388,608)
  K=  16: RadiK=   3.41 ms | PyTorch=   1.03 ms | Speedup= 0.30x
  K=  32: RadiK=   3.33 ms | PyTorch=   1.03 ms | Speedup= 0.31x
  K=  64: RadiK=   3.52 ms | PyTorch=   1.06 ms | Speedup= 0.30x
  K= 128: RadiK=   3.41 ms | PyTorch=   1.03 ms | Speedup= 0.30x
  K= 256: RadiK=   3.43 ms | PyTorch=   1.0

In [5]:
#! /usr/bin/env python3

# BATCH BENCHMARKS


import csv
import os.path
from typing import List
import sys
import numpy as np

script_dir = '/content/radik/eval'
sys.path.insert(0, script_dir)

import libs.common as common

common.BASE_DIR = os.path.abspath('/content/radik')
common.PLOT_DIR = os.path.abspath(os.path.join(script_dir, '..', 'plot'))
os.makedirs(common.PLOT_DIR, exist_ok=True)

common.RADIK_BIN = os.path.join(common.BASE_DIR, "radik", "test_radik.out")
common.BITONIC_BIN = os.path.join(common.BASE_DIR, "bitonic", "compareTopKAlgorithms")
common.BLOCKSELECT_BIN = os.path.join(common.BASE_DIR, "blockselect", "test_block_select.out")
common.PQ_BIN = os.path.join(common.BASE_DIR, "radik", "test_PQ.out")
common.ABLATION_BIN = os.path.join(common.BASE_DIR, "radik", "test_ablation.out")

from libs.algos import ALGOS, Algo


In [6]:
START = 16
END = 23

BATCH = 16
K = 512


def fn(algo: Algo):
    return ALGOS[algo][1]


def test_case(N: int) -> List[float]:
    res = []
    res.append(fn(Algo.RADIK)(BATCH, N, K))
    return res


def main():
    res = []
    for N in range(START, END):
        print(f"Eval: BATCH={BATCH}, N={N}, K={K}")
        res.append(test_case(N))
    res = np.array(res)
    res = res.T
    print("Done")

    # Print results as a table
    print("\n" + "="*80)
    print("RESULTS: Input Length Variation (BATCH=16, K=512)")
    print("="*80)
    print(f"{'N':<12} {'Latency (ms)':<15}")
    print("-"*80)

    for i, N in enumerate(range(START, END)):
        n_label = f"2^{N}"
        latency = res[0][i]
        print(f"{n_label:<12} {latency:<15.2f}")

    print("="*80)

main()

Eval: BATCH=16, N=16, K=512
Eval: BATCH=16, N=17, K=512
Eval: BATCH=16, N=18, K=512
Eval: BATCH=16, N=19, K=512
Eval: BATCH=16, N=20, K=512
Eval: BATCH=16, N=21, K=512
Eval: BATCH=16, N=22, K=512
Done

RESULTS: Input Length Variation (BATCH=16, K=512)
N            Latency (ms)   
--------------------------------------------------------------------------------
2^16         3.03           
2^17         3.21           
2^18         3.20           
2^19         3.32           
2^20         3.81           
2^21         4.87           
2^22         5.97           


In [7]:
START = 4
END = 11

BATCH = 16
N = 22


def fn(algo: Algo):
    return ALGOS[algo][1]


def test_case(K: int) -> List[float]:
    res = []
    res.append(fn(Algo.RADIK)(BATCH, N, K))
    return res


def main():
    res = []
    for K_order in range(START, END):
        K = 1 << K_order
        print(f"Eval: BATCH={BATCH}, N={N}, K={K}")
        res.append(test_case(K))
    res = np.array(res)
    res = res.T
    print("Done")

    # Print results as a table
    print("\n" + "="*80)
    print("RESULTS: k Variation (BATCH=16, N=2^22)")
    print("="*80)
    print(f"{'K':<12} {'Latency (ms)':<15}")
    print("-"*80)

    for i, K_order in enumerate(range(START, END)):
        k_value = 1 << K_order
        k_label = f"{k_value}"
        latency = res[0][i]
        print(f"{k_label:<12} {latency:<15.2f}")

    print("="*80)

main()

Eval: BATCH=16, N=22, K=16
Eval: BATCH=16, N=22, K=32
Eval: BATCH=16, N=22, K=64
Eval: BATCH=16, N=22, K=128
Eval: BATCH=16, N=22, K=256
Eval: BATCH=16, N=22, K=512
Eval: BATCH=16, N=22, K=1024
Done

RESULTS: k Variation (BATCH=16, N=2^22)
K            Latency (ms)   
--------------------------------------------------------------------------------
16           6.37           
32           6.17           
64           6.15           
128          6.16           
256          7.11           
512          6.41           
1024         6.56           


In [8]:
START = 0
END = 7

N = 22
K = 512


def fn(algo: Algo):
    return ALGOS[algo][1]


def test_case(BATCH: int) -> List[float]:
    res = []
    res.append(fn(Algo.RADIK)(BATCH, N, K))
    return res


def main():
    res = []
    for B_order in range(START, END):
        BATCH = 1 << B_order
        print(f"Eval: BATCH={BATCH}, N={N}, K={K}")
        res.append(test_case(BATCH))
    res = np.array(res)
    res = res.T
    print("Done")

    # Print results as a table
    print("\n" + "="*80)
    print("RESULTS: Batch Size Variation (N=2^22, K=512)")
    print("="*80)
    print(f"{'Batch Size':<12} {'Latency (ms)':<15}")
    print("-"*80)

    for i, B_order in enumerate(range(START, END)):
        batch_value = 1 << B_order
        batch_label = f"{batch_value}"
        latency = res[0][i]
        print(f"{batch_label:<12} {latency:<15.2f}")

    print("="*80)

main()

Eval: BATCH=1, N=22, K=512
Eval: BATCH=2, N=22, K=512
Eval: BATCH=4, N=22, K=512
Eval: BATCH=8, N=22, K=512
Eval: BATCH=16, N=22, K=512
Eval: BATCH=32, N=22, K=512
Eval: BATCH=64, N=22, K=512
Done

RESULTS: Batch Size Variation (N=2^22, K=512)
Batch Size   Latency (ms)   
--------------------------------------------------------------------------------
1            3.32           
2            3.49           
4            3.79           
8            4.71           
16           6.14           
32           9.24           
64           18.68          
